In [ ]:
import os
import cv2
import tifffile 
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.interpolate import RegularGridInterpolator
import matplotlib.animation as animation
import csv

### Path to the folder with the flow data
paper_data_path = "M:\\Shared\\Paramecium_functions_paper\\flow_analysis"

### Choose the type of data to focus on
type = "only_doublets\\deciliated_cells"

### Specify the pixel size and frame rate 
pixel_size = 0.325 # um
fps        = 33  

### Reading the aligned flow data paths for this condition
flow_folder_path = paper_data_path + "\\" + type + "\\" + "piv_as_npy\\"
flow_aligned_paths = []
flow_file_name = "_aligned_piv.npz"

for file in os.listdir(flow_folder_path):
    if file.endswith(flow_file_name):
        flow_aligned_paths.append(flow_folder_path + file)

### Reading the cell mask paths for this condition
masks_folder_path = paper_data_path + "\\" + type + "\\" + "masks\\"
masks_paths = []
masks_file_name = "_masks.tif"


for file in os.listdir(masks_folder_path):
    if file.endswith(masks_file_name):
        masks_paths.append(masks_folder_path + file)

cell_ids = []

### Extracting cell IDs for this condition
for path in masks_paths:
    filename = path.split("\\")[-1]
    cell_id = filename.split("_")[0] # + "_" + filename.split("_")[1]

    cell_ids.append(cell_id)


for cell_id in tqdm(cell_ids, desc="Analyzing cells"):

    output_path = paper_data_path + "\\" + type + "\\analysis\\" + cell_id + "\\"

    if not os.path.exists(output_path):
        os.makedirs(output_path)


    flow_aligned = np.load(paper_data_path + "\\"+ type + "\\piv_as_npy\\" + cell_id + "_filt_aligned_piv.npz", allow_pickle=True)
    cell_mask = tifffile.imread(paper_data_path + "\\"+ type + "\\masks\\" + cell_id + "_filt_masks.tif")

    ### Reading the grid coordinates and velocities in arrays for convenience
    ###----------------------------------------------------------------------
    ### The grid points in pixels
    x_grid     = flow_aligned['x_grid']
    y_grid     = flow_aligned['y_grid']
    ### The flow velocities in grid units prior to alignment (pixels per frame)
    all_u      = flow_aligned['all_u']
    all_v      = flow_aligned['all_v']
    ### The flow velocities in grid units after alignment (pixels per frame)
    u_aligned  = flow_aligned['u_aligned']
    v_aligned  = flow_aligned['v_aligned']

    ### Extracting other useful parameters
    ###----------------------------------------------------------------------
    duration_initial = flow_aligned['u_aligned'].shape[2]   # The duration of the timelapse (frames)
    grid_dim_x       = flow_aligned['u_aligned'].shape[0]   # The number of grid points along x
    grid_dim_y       = flow_aligned['u_aligned'].shape[1]   # The number of grid points along y

    duration_sec_initial = duration_initial / fps     # The duration of the timelapse (seconds)
    image_dim_x          = cell_mask.shape[2]  # The number of pixels along x
    image_dim_y          = cell_mask.shape[1]  # The number of pixels along y

    grid_size_x = (image_dim_x // grid_dim_x) # The size of the grid unit in pixels along x
    grid_size_y = (image_dim_y // grid_dim_y) # The size of the grid unit in pixels along y

    ### Printing the parameters for verification
    print(f"{'Pixel size':<22}: {pixel_size} μm/pixel")
    print(f"{'Frame rate':<22}: {fps} frames/sec")
    print(f"{'Grid dimensions':<22}: {grid_dim_x} x {grid_dim_y} grid units")
    print(f"{'Timelapse duration':<22}: {duration_initial} frames")
    print(f"{'Grid size':<22}: {grid_size_x} x {grid_size_y} pixels")
    print(f"{'In physical units':<22}: {image_dim_x} x {image_dim_y} μm")
    print(f"{'Timelapse duration':<22}: {np.round(duration_sec_initial, 2)} seconds")

    ###---------------------------------------------------------------------------------------------------------------
    ### Calculating the flow velocity magnitude in physical units (um/s)
    norm_velocity = np.hypot(u_aligned, v_aligned) /pixel_size * fps

    ### Mild temporal averaging of the flow velocity magnitude to reduce noise
    def temporal_average(data, dt):
        # Get the shape
        h, w, t = data.shape
        
        # Number of output time steps after applying window
        new_t = t - dt + 1
        
        # Initialize output array
        averaged_data = np.empty((h, w, new_t))
        
        # Perform the moving average
        for i in range(new_t):
            averaged_data[:, :, i] = np.mean(data[:, :, i:i+dt], axis=2)
        
        return averaged_data

    ### Applying temporal averaging
    dt_average = 15 # frames
    norm_vel_smoothed = temporal_average(norm_velocity, dt_average)
    u_smooth = temporal_average(u_aligned, dt_average)
    v_smooth = temporal_average(v_aligned, dt_average)

    ### Updating variables after smoothing (if no smoothing (dt_average = 1), this is meant to allow keeping the same variable names for the rest of the code)
    norm_velocity = norm_vel_smoothed
    duration = norm_velocity.shape[2]
    duration_sec = duration / fps  
    u = u_smooth
    v = v_smooth
    ###---------------------------------------------------------------------------------------------------------------

    ### Saving parameters for convenience and troubleshooting
    csv_timelapse_parametrs_filename = cell_id + "_timelapse_parameters.csv"

    with open(output_path + csv_timelapse_parametrs_filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
    
        writer.writerow(["Parameter", "Value"])
        writer.writerow(["Pixel size (um/pixel)", pixel_size])
        writer.writerow(["Frame rate (fps)", fps])
        writer.writerow(["Image dimensions x (um)", image_dim_x])
        writer.writerow(["Image dimensions y (um)", image_dim_y])
        writer.writerow(["Grid dimensions x (grid units)", grid_dim_x])
        writer.writerow(["Grid dimensions y (grid units)", grid_dim_y])
        writer.writerow(["Grid size x (pixels)", grid_size_x])
        writer.writerow(["Grid size y (pixels)", grid_size_y])
        writer.writerow(["Timelapse duration (frames)", duration_initial])    
        writer.writerow(["Timelapse duration (sec)", np.round(duration_sec_initial, 2)])
        writer.writerow(["Temporal averaging window (frames)", dt_average])
        writer.writerow(["Smoothed timelapse duration (frames)", duration])    
        writer.writerow(["Smoothed timelapse duration (sec)", np.round(duration_sec, 2)])
    print(f"Data successfully saved to {csv_timelapse_parametrs_filename}")

    ###---------------------------------------------------------------------------------------------------------------
    ### Input parameters
    ###------------------------------------------------------------------------------
    gif_label = "False"  # Set gif_label to "True" to additionally save a GIF
    step      = 2        # If there is any need of downsampling the quiver plot, set the step size here.
    frame_id  = 0        # Choose frame to plot
    u_max     = 1200     # Set the maximum velocity for the colorbar (um/s)
    scale     = 1/10      # Scale for the quiver plot
    width     = 0.002    # Width for the quiver plot

    ### Convert the PIV grid coordinates to physical units (um)
    ###------------------------------------------------------------------------------
    x_um = x_grid * pixel_size
    y_um = y_grid * pixel_size

    # Pre-convert the full arrays to handle orientation signs safely
    u_um_all = u * pixel_size
    v_um_all = v * pixel_size  # Fixed vertical orientation index


    u_um = u_um_all[:, :, frame_id]
    v_um = v_um_all[:, :, frame_id]

    ### Downsample (optional) for the visualization of the flow vectors
    x_um_ds = x_um[::step, ::step]
    y_um_ds = y_um[::step, ::step]
    u_um_ds = u_um[::step, ::step]
    v_um_ds = v_um[::step, ::step]

    ### Plot extent in physical coordinates
    x_min = np.nanmin(x_um)
    x_max = np.nanmax(x_um)
    y_min = np.nanmin(y_um)
    y_max = np.nanmax(y_um)

    ###---------------------------------------------------------------------------------------------------------------
    ### Saving the processed data and parameters to be able to directly change the plot in separate scripts
    npz_velocity_filename = os.path.join(output_path, f"{cell_id}_smoothed_velocity_field.npz")
    csv_velocity_filename = os.path.join(output_path, f"{cell_id}_plotting_velocity_parameters.csv")

    ## Saving velocity and grid arrays to npz
    np.savez_compressed(
        npz_velocity_filename, 
        norm_velocity=norm_velocity, 
        u=u, 
        v=v,
        u_um_all=u_um_all,
        v_um_all=v_um_all,
        x_um_ds=x_um_ds,
        y_um_ds=y_um_ds,
        u_um_ds=u_um_ds,
        v_um_ds=v_um_ds
    )

    ## Saving saving plotting paramters to csv
    with open(csv_velocity_filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Parameter", "Value"]) # Header row
        writer.writerow(["cell_id", cell_id])
        writer.writerow(["frame_id", frame_id])
        writer.writerow(["x_min", x_min])
        writer.writerow(["x_max", x_max])
        writer.writerow(["y_min", y_min])
        writer.writerow(["y_max", y_max])
        writer.writerow(["u_max", u_max])
        writer.writerow(["scale", scale])
        writer.writerow(["width", width])
        writer.writerow(["step", step])
        writer.writerow(["duration", duration])

    print(f"Successfully saved binary matrices to: {npz_velocity_filename}")
    print(f"Successfully saved configuration parameters to: {csv_velocity_filename}")
    ###---------------------------------------------------------------------------------------------------------------
    ### Plotting the flow field for the selected frame
    cmap = plt.cm.viridis.copy()
    cmap.set_bad("black")
    im_data = np.ma.masked_invalid(norm_velocity[:, :, frame_id])

    fig, ax = plt.subplots(figsize=(7, 7))

    # Fixed extent format to maintain natural top-down coordinate space
    im = ax.imshow(
        im_data,
        cmap=cmap,
        extent=[x_min, x_max, y_min, y_max],
        origin="upper",
    )

    # Add a colorbar with a label
    im.set_clim(0, u_max)
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im, label="Velocity (\u03BCm/s)", cax=cax)

    # Clean quiver layout matching the real tracking matrix
    ax.quiver(
        x_um_ds,
        y_um_ds,
        u_um_ds,
        v_um_ds,
        color="white",
        scale=scale,
        width=width,

        scale_units="xy",
        angles="xy",
    )

    ax.set_xlabel("x (\u03BCm)")
    ax.set_ylabel("y (\u03BCm)")
    ax.axis("on")
    plt.show()

    # Save GIF only if gif_label is "True"
    if gif_label == "True":
        import matplotlib.animation as animation

        output_gif = paper_data_path + "\\" + type + "\\analysis\\" + cell_id + "_flow_animation_um.gif"
        n_frames = duration

        fig, ax = plt.subplots(figsize=(6, 6))

        im = ax.imshow(
            np.ma.masked_invalid(norm_velocity[:, :, 0]),
            cmap=cmap,
            extent=[x_min, x_max, y_min, y_max],  # Fixed extent format
            origin="upper",
        )
        im.set_clim(0, u_max)
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.05)
        plt.colorbar(im, label="Velocity (\u03BCm/s)", cax=cax)

        quiver = ax.quiver(
            x_um_ds,
            y_um_ds,
            u_um_all[::step, ::step, 0],
            v_um_all[::step, ::step, 0], 
            color="white",
            scale=scale,
            width=width,
            scale_units="xy",
            angles="xy",
        )

        ax.set_xlabel("x (\u03BCm)")
        ax.set_ylabel("y (\u03BCm)")
        ax.axis("on")

        def update(frame_index):
            im.set_data(np.ma.masked_invalid(norm_velocity[:, :, frame_index]))
            quiver.set_UVC(
                u_um_all[::step, ::step, frame_index],
                v_um_all[::step, ::step, frame_index],  
            )
            ax.set_title(f"Frame {frame_index + 1}/{n_frames}")
            return im, quiver

        anim = animation.FuncAnimation(
            fig,
            update,
            frames=n_frames,
            interval=100,
            blit=False,
        )
        output_gif = output_path + cell_id + "_flow_animation_um.gif"
        anim.save(output_gif, writer=animation.PillowWriter(fps=10))
        plt.close(fig)
        print(f"Saved GIF to {output_gif}")

    ### Computing and ploting vorticity field
    ###---------------------------------------------------------------------------------------------------------------
    ### Set to "True" to generate and save a gif for the whole timelapse
    vorticity_gif = "False "
    fps_gif = 10

    x_um_vec = x_um[0, :]
    y_um_vec = y_um[:, 0]

    # Keep the image orientation consistent with the other cell plots
    extent_um = [x_um_vec.min(), x_um_vec.max(), y_um_vec.min(), y_um_vec.max()]

    n_frames = duration

    omega_all = np.empty((u.shape[0], u.shape[1], n_frames), dtype=float)

    for t in range(n_frames):
        # Convert to physical units for correct vorticity in 1/s
        u_phys = u[:, :, t] 
        v_phys = v[:, :, t] 

        dv_dy, dv_dx = np.gradient(v_phys, y_um_vec, x_um_vec)
        du_dy, du_dx = np.gradient(u_phys, y_um_vec, x_um_vec)
        omega_all[:, :, t] = dv_dx - du_dy

    omega_lim = np.nanpercentile(np.abs(omega_all), 99)
    print(omega_lim)
    ###---------------------------------------------------------------------------------------------------------------
    # --- Show first frame ---
    fig, ax = plt.subplots(figsize=(7, 7))
    im = ax.imshow(
        np.ma.masked_invalid(omega_all[:, :, 0]),
        cmap="RdBu_r",
        extent=extent_um,
        origin="upper",
        vmin=-omega_lim,
        vmax=omega_lim,
    )

    ax.quiver(
        x_um_ds,
        y_um_ds,
        u_um_ds,
        v_um_ds,
        color="black",
        scale=scale,
        width=width,
        scale_units="xy",
        angles="xy",
    )

    plt.colorbar(im, ax=ax, label="vorticity (1/s)")
    ax.set_xlabel("x (μm)")
    ax.set_ylabel("y (μm)")
    ax.set_title("Vorticity map, frame 1")
    plt.show()

    # --- Save GIF ---
    if vorticity_gif == "True" and n_frames > 1:
        fig, ax = plt.subplots(figsize=(7, 7))

        im = ax.imshow(
            np.ma.masked_invalid(omega_all[:, :, 0]),
            cmap="RdBu_r",
            extent=extent_um,
            origin="upper",
            vmin=-omega_lim,
            vmax=omega_lim,
        )
        plt.colorbar(im, ax=ax, label="vorticity (1/s)")
        ax.set_xlabel("x (μm)")
        ax.set_ylabel("y (μm)")

        quiver = ax.quiver(
            x_um_ds,
            y_um_ds,
            u_um_all[::step, ::step, 0],
            v_um_all[::step, ::step, 0],
            color="black",
            scale=scale,
            width=width,
            scale_units="xy",
            angles="xy",
        )

        def update(frame_index):
            im.set_data(np.ma.masked_invalid(omega_all[:, :, frame_index]))
            quiver.set_UVC(
                u_um_all[::step, ::step, frame_index],
                v_um_all[::step, ::step, frame_index],
            )
            ax.set_title(f"Vorticity map, frame {frame_index + 1}/{n_frames}")
            return im, quiver

        anim = animation.FuncAnimation(
            fig,
            update,
            frames=n_frames,
            interval=100,
            blit=False,
        )

        output_gif = output_path + cell_id + "_vorticity_animation_um.gif"
        anim.save(output_gif, writer=animation.PillowWriter(fps=fps_gif))
        plt.close(fig)
        print(f"Saved vorticity GIF to {output_gif}")
    ###---------------------------------------------------------------------------------------------------------------
    ### Saving the vorticity field information 
    npz_vorticity_filename = os.path.join(output_path, f"{cell_id}_vorticity_field.npz")
    csv_vorticity_filename = os.path.join(output_path, f"{cell_id}_plotting_vorticity_parameters.csv")

    np.savez_compressed(
        npz_vorticity_filename, 
        omega_all=omega_all,
        dv_dy=dv_dy,
        dv_dx=dv_dx,
        du_dy=du_dy,
        du_dx=du_dx,
        x_um_vec=x_um_vec,
        y_um_vec=y_um_vec,
        u_um_all=u_um_all,
        v_um_all=v_um_all,
        x_um_ds=x_um_ds,
        y_um_ds=y_um_ds,
        u_um_ds=u_um_ds,
        v_um_ds=v_um_ds
    )

    with open(csv_vorticity_filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Parameter", "Value"])
        writer.writerow(["cell_id", cell_id])
        writer.writerow(["x_min", x_min])
        writer.writerow(["x_max", x_max])
        writer.writerow(["y_min", y_min])
        writer.writerow(["y_max", y_max])
        writer.writerow(["omega_lim", omega_lim])
        writer.writerow(["scale", scale])
        writer.writerow(["width", width])
        writer.writerow(["step", step])
        writer.writerow(["n_frames", duration])
        writer.writerow(["fps_gif", fps_gif])

    print(f"Successfully appended vorticity arrays to: {npz_vorticity_filename}")
    print(f"Successfully saved configuration parameters to: {csv_vorticity_filename}")

    ### Finding the cell contour and axis
    ###---------------------------------------------------------------------------------------------------------------
    ### Choose frame to focus
    frame_to_plot = 0
    mask = cell_mask[frame_to_plot]

    ### Contour detection
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    fig, ax = plt.subplots()

    ### Fixing y dimension of matplotlib to match previous data visualization
    max_y_um = image_dim_y * pixel_size
    max_x_um = image_dim_x * pixel_size

    x_um_mask = np.linspace(0, image_dim_x * pixel_size, image_dim_x)
    y_um_mask = np.linspace(max_y_um, 0, image_dim_y)

    ax.imshow(mask, cmap="gray", extent=[x_um_mask.min(), x_um_mask.max(), y_um_mask.min(), y_um_mask.max()], origin="upper")
    ax.contourf(x_um_mask, y_um_mask, mask, levels=[0.5, 1], colors="magenta", alpha=1)

    if contours and len(contours[0]) >= 5:
        ### Select the area corresponding to the cell
        largest_contour = max(contours, key=cv2.contourArea)

        ### Ellipse fitting
        ellipse = cv2.fitEllipse(largest_contour)

        (cx, cy), (raw_dim1, raw_dim2), deg_angle = ellipse
        center = np.array([cx, image_dim_y - cy]) * pixel_size
        # Ensure major axis is always the maximum length
        if raw_dim1 < raw_dim2:
            ma = raw_dim2
            mi = raw_dim1
            # If we flip the axes, we must shift the reference angle by 90 degrees
            deg_angle = (deg_angle + 90) % 360
        else:
            ma = raw_dim1
            mi = raw_dim2

        major_length = ma * pixel_size
        minor_length = mi * pixel_size

        center_x = cx * pixel_size
        center_y = max_y_um - (cy * pixel_size)  # Direct Matplotlib Y-conversion
        angle = np.deg2rad(deg_angle)

        ### Plotting ellipse
        ellipse_poly = cv2.ellipse2Poly((int(cx), int(cy)), (int(ma / 2), int(mi / 2)), int(deg_angle), 0, 360, 1)
        ellipse_poly = np.array(ellipse_poly) * pixel_size
        ellipse_poly[:, 1] = max_y_um - ellipse_poly[:, 1]  # Invert Y map
        ax.plot(ellipse_poly[:, 0], ellipse_poly[:, 1], "r--", linewidth=1.5, label="Fitted Ellipse")

        ### Plotting ellipse axes
        u_major = np.array([np.cos(angle), -np.sin(angle)])
        u_minor = np.array([np.cos(angle + np.pi/2), -np.sin(angle + np.pi/2)])

        half_major = major_length / 2
        half_minor = minor_length / 2

        c_point = np.array([center_x, center_y])

        major_pts = np.array([c_point + half_major * u_major, c_point - half_major * u_major])
        minor_pts = np.array([c_point + half_minor * u_minor, c_point - half_minor * u_minor])

        ax.plot(major_pts[:, 0], major_pts[:, 1], "y-", linewidth=2, label="Major Axis")
        ax.plot(minor_pts[:, 0], minor_pts[:, 1], "c-", linewidth=2, label="Minor Axis")
        ax.plot(center_x, center_y, "ro", markersize=5, label="Center")

    ax.set_xlabel("x (μm)")
    ax.set_ylabel("y (μm)")
    ax.set_aspect('equal')
    plt.savefig(os.path.join(output_path, f"{cell_id}_ellipse_fit.png"), dpi=300)
    plt.show()

    print("Major axis length (μm):", major_length)
    print("Minor axis length (μm):", minor_length)
    ###---------------------------------------------------------------------------------------------------------------
    csv_cell_ellipse_filename = os.path.join(output_path, f"{cell_id}_ellipse_parameters.csv")

    ### Saving results
    with open(csv_cell_ellipse_filename, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        
        # Optional visual spacer row
        writer.writerow(["# --- Fitted Ellipse Geometry Parameters ---", ""])
        
        # Save the absolute minimum required parameters vertically
        writer.writerow(["center_x", center_x])
        writer.writerow(["center_y", center_y])
        writer.writerow(["major_length", major_length])
        writer.writerow(["minor_length", minor_length])
        writer.writerow(["angle_rad", angle])  # Saving the radian angle directly
        writer.writerow(["max_y_um", max_y_um])



    print(f"Geometry metrics saved in: {csv_cell_ellipse_filename}")

    ### Generating radial lines from the cell center for flow analysis
    ###---------------------------------------------------------------------------------------------------------------
    ### Input parameters for radial line generation
    ###---------------------------------------------
    N             = 3*4     # Number of radial lines (Note:Must be a multiple of 4 for minor axis alignment!)
    max_radius_um = 200.0   # Length of each line in um
    num_pts       = 100     # Number of sample points along each line

    # Useful coordinates
    c_point = np.array([center_x, center_y])
    alpha = np.deg2rad(deg_angle)
    cos_a, sin_a = np.cos(alpha), np.sin(alpha)

    # Standard rotation matrix that preserves the axes direction aligned with the axes detected
    R = np.array([[cos_a, sin_a], 
                [-sin_a, cos_a]])

    ### Creating the line segments
    r_segments = np.linspace(0, max_radius_um, num_pts)

    ### Generate angles 
    local_angles = np.linspace(0, 2 * np.pi, N, endpoint=False)

    # Generates the rotated lines
    radial_lines_x = {}
    radial_lines_y = {}

    for angle in local_angles:
  
        total_angle = angle + alpha
        
        global_vector = np.array([np.cos(total_angle), -np.sin(total_angle)])

        x_line_coords = c_point[0] + global_vector[0] * r_segments
        y_line_coords = c_point[1] + global_vector[1] * r_segments
        
        deg = int(np.round(np.degrees(angle)))
        radial_lines_x[deg] = x_line_coords
        radial_lines_y[deg] = y_line_coords

    print(f"Generated {len(radial_lines_x)} radial lines.")
    print(f"Available line angles: {list(radial_lines_x.keys())}°")

    ### Plotting the lines for verification
    fig, ax = plt.subplots(figsize=(7, 7))

    ax.imshow(mask, cmap="gray", extent=[x_um_mask.min(), x_um_mask.max(), y_um_mask.min(), y_um_mask.max()], origin="upper", alpha=0.5)

    colors = plt.cm.jet(np.linspace(0, 1, N))

    for i, deg in enumerate(radial_lines_x.keys()):
        x_pts = radial_lines_x[deg]

        y_pts = radial_lines_y[deg] 
        
        if deg == 0:
            label_text = f"Major Axis - ({deg}°)"
            linewidth, linestyle = 2.5, "-"
        elif deg == 180:
            label_text = f"Major Axis + ({deg}°)"
            linewidth, linestyle = 2.5, "-"
        elif deg == 90:
            label_text = f"Minor Axis + ({deg}°)"
            linewidth, linestyle = 2.5, "-"
        elif deg == 270:
            label_text = f"Minor Axis - ({deg}°)"
            linewidth, linestyle = 2.5, "-"
        else:
            label_text = f"Ray {deg}°"
            linewidth, linestyle = 1.2, ":"
        
        ax.plot(x_pts, y_pts, color=colors[i], linewidth=linewidth, 
                linestyle=linestyle, label=label_text)

    ax.plot(c_point[0], c_point[1], "ro", markersize=6, label="Cell Center")

    ax.set_xlabel("x (μm)")
    ax.set_ylabel("y (μm)")
    ax.set_title(f"Generated Radial Profile Lines (N={N})")
    ax.set_aspect("equal")

    ax.legend(loc="upper left", bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()
    plt.savefig(os.path.join(output_path, f"{cell_id}_radial_lines.png"), dpi=300)
    plt.show()

    ###---------------------------------------------------------------------------------------------------------------
    ### New Angular Sector Parameters
    ###---------------------------------------------
    angular_window_deg = 30.0    # Total angular width of the wedge sector in degrees (e.g., +/- 7.5 deg)
    num_angular_pts    = 11     # Number of angular sampling steps within the window

    ### Generating the baseline centerline segments
    radial_lines_x = {}
    radial_lines_y_plot = {} 

    for angle in local_angles:
        total_angle = angle + alpha
        
        global_vector = np.array([np.cos(total_angle), -np.sin(total_angle)])
        x_line_coords = center_x + global_vector[0] * r_segments
        y_line_plot_coords = center_y + global_vector[1] * r_segments

        deg = int(np.round(np.degrees(angle)))
        radial_lines_x[deg] = x_line_coords
        radial_lines_y_plot[deg] = y_line_plot_coords

    ### Interpolating the velocity grid map
    vel_frame = norm_velocity[:, :, frame_id]

    if y_um_vec[0] > y_um_vec[-1]:
        y_interp_grid = np.flip(y_um_vec)
        vel_interp_grid = np.flipud(vel_frame)
    else:
        y_interp_grid = y_um_vec
        vel_interp_grid = vel_frame

    x_interp_grid = x_um_vec

    vel_interp = RegularGridInterpolator(
        (y_interp_grid, x_interp_grid), 
        vel_interp_grid, 
        bounds_error=False, 
        fill_value=np.nan
    )

    ### Plotting setup
    cmap = plt.cm.viridis.copy()
    cmap.set_bad("black")
    im_data = np.ma.masked_invalid(vel_frame)

    fig, (ax_map, ax_profile) = plt.subplots(1, 2, figsize=(19, 7))

    # Panel A: Flow Map 
    im = ax_map.imshow(
        im_data,
        cmap=cmap,
        extent=[x_min, x_max, y_min, y_max],
        origin="upper", alpha=0.5, zorder=1
    )
    im.set_clim(0, u_max)

    ax_map.quiver(
        x_um_ds, y_um_ds, u_um_ds, v_um_ds,
        color="white", scale=scale, width=width,
        scale_units="xy", angles="xy", zorder=4
    )

    # Extracting the profiles 
    colors = plt.cm.jet(np.linspace(0, 1, N))

    csv_rows = []

    for i, deg in enumerate(radial_lines_x.keys()):
        

        x_pts = radial_lines_x[deg]
        y_pts_plot = radial_lines_y_plot[deg]
        
        central_angle_rad = np.deg2rad(deg) + alpha
        
        half_window_rad = np.deg2rad(angular_window_deg) / 2
        theta_segments = np.linspace(central_angle_rad - half_window_rad, 
                                    central_angle_rad + half_window_rad, 
                                    num_angular_pts)
        

        r_mesh, theta_mesh = np.meshgrid(r_segments, theta_segments)
        
        x_wedge = center_x + np.cos(theta_mesh) * r_mesh
        y_wedge = center_y - np.sin(theta_mesh) * r_mesh  

        sample_points = np.vstack([y_wedge.ravel(), x_wedge.ravel()]).T
        sampled_velocity_flat = vel_interp(sample_points)
        
        sampled_velocity_grid = sampled_velocity_flat.reshape(num_angular_pts, len(r_segments))
        
        sampled_velocity = np.nanmean(sampled_velocity_grid, axis=0)
        
        if deg == 0:
            label_text = f"Major Axis - ({deg}°)"
            linewidth, linestyle = 2.5, "-"
        elif deg == 180:
            label_text = f"Major Axis + ({deg}°)"
            linewidth, linestyle = 2.5, "-"
        elif deg == 90:
            label_text = f"Minor Axis + ({deg}°)"
            linewidth, linestyle = 2.5, "--"  
        elif deg == 270:
            label_text = f"Minor Axis - ({deg}°)"
            linewidth, linestyle = 2.5, "--"
        else:
            label_text = f"Ray {deg}°"
            linewidth, linestyle = 1.2, ":"
            
        # For plotting the lines for visual inspection
        poly_x = np.concatenate(([center_x], x_wedge[0, :], x_wedge[:, -1], x_wedge[-1, :][::-1]))
        poly_y = np.concatenate(([center_y], y_wedge[0, :], y_wedge[:, -1], y_wedge[-1, :][::-1]))
        
        ax_map.fill(poly_x, poly_y, color=colors[i], alpha=0.15, zorder=2)
        
        ax_map.plot(x_wedge[0, :], y_wedge[0, :], color=colors[i], linewidth=0.8, linestyle="--", alpha=0.4, zorder=2)
        ax_map.plot(x_wedge[-1, :], y_wedge[-1, :], color=colors[i], linewidth=0.8, linestyle="--", alpha=0.4, zorder=2)

        ax_map.plot(x_pts, y_pts_plot, color=colors[i], linewidth=linewidth, linestyle=linestyle, zorder=3)
        
        ax_profile.plot(r_segments, sampled_velocity, color=colors[i], 
                        linewidth=linewidth, linestyle=linestyle, label=label_text)


        ### Saving data:
        for r_idx, radius in enumerate(r_segments):
            csv_rows.append({
                "Frame_ID": frame_id,
                "Angle_Deg": deg,
                "Distance_um": radius,
                "Velocity_ums": sampled_velocity[r_idx]
            })


    # Plotting the center of the cell
    ax_map.plot(center_x, center_y, "ro", markersize=6, label="Cell Center", zorder=5)

    ### Append colorbar axis 
    divider = make_axes_locatable(ax_map)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im, label="Velocity (μm/s)", cax=cax)

   
    ax_map.set_xlim(x_min, x_max)
    ax_map.set_ylim(y_min, y_max)
    ax_map.set_xlabel("x (μm)")
    ax_map.set_ylabel("y (μm)")
    ax_map.set_title(f"Averaging Wedges over PIV Grid (Frame {frame_id})")
    ax_map.axis("on")

 
    ax_profile.set_xlabel("Distance from Center (μm)")
    ax_profile.set_ylabel("Velocity Amplitude (μm/s)")
    ax_profile.set_title(f"Sector-Averaged Radial Flow Profiles ($\Delta\\theta$={angular_window_deg}°)")
    ax_profile.grid(True, linestyle="--", alpha=0.5)
    ax_profile.set_ylim(0, u_max)  
    ax_profile.set_xlim(0, max_radius_um)
    ax_profile.legend(loc="upper left", bbox_to_anchor=(1.05, 1), borderaxespad=0.)

    plt.tight_layout()
    plt.savefig(output_path + f"\\{cell_id}_radial_profiles_frame_{frame_id}.png", dpi=300)
    plt.show()


    ###---------------------------------------------------------------------------------------------------------------
    ### Saving data for flowprofie analysis
    npz_profiles_filename = os.path.join(output_path, f"{cell_id}_radial_profiles_analysis.npz")
    csv_profiles_filename = os.path.join(output_path, f"{cell_id}_radial_profiles_analysis.csv")

    np.savez_compressed(
        npz_profiles_filename,
        norm_velocity=norm_velocity,
        x_um_ds=x_um_ds,
        y_um_ds=y_um_ds,
        u_um_ds=u_um_ds,
        v_um_ds=v_um_ds,
        r_segments=r_segments,
        local_angles=local_angles,
        x_um_vec=x_um_vec,
        y_um_vec=y_um_vec
    )

    with open(csv_profiles_filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        
        writer.writerow(["Parameter", "Value"])
        writer.writerow(["cell_id", cell_id])
        writer.writerow(["frame_id", frame_id])
        writer.writerow(["angular_window_deg", angular_window_deg])
        writer.writerow(["num_angular_pts", num_angular_pts])
        writer.writerow(["alpha", alpha])
        writer.writerow(["center_x", center_x])
        writer.writerow(["center_y", center_y])
        writer.writerow(["x_min", x_min])
        writer.writerow(["x_max", x_max])
        writer.writerow(["y_min", y_min])
        writer.writerow(["y_max", y_max])
        writer.writerow(["u_max", u_max])
        writer.writerow(["max_radius_um", max_radius_um])
        writer.writerow(["scale", scale])
        writer.writerow(["width", width])
        writer.writerow(["N", N])
        writer.writerow([]) # Separator spacer row
        
        # Append the profile values
        writer.writerow(["Frame_ID", "Angle_Deg", "Distance_um", "Velocity_ums"])
        for row in csv_rows:
            writer.writerow([row["Frame_ID"], row["Angle_Deg"], row["Distance_um"], row["Velocity_ums"]])

    print(f"Data successfully saved to : {npz_profiles_filename}\n and: {csv_profiles_filename}")
  
